In [ ]:
!pip -q install ImageHash

Set-up bridge between Google Colab and Google Drive.

In [ ]:
# --- Mount Google Drive cleanly ---
from google.colab import drive

# Unmount if already mounted
try:
    drive.flush_and_unmount()
except Exception:
    pass

# Mount
drive.mount("/content/drive", force_remount=True)

# Free space check on Drive
import shutil, os, pathlib
stat = shutil.disk_usage("/content/drive/MyDrive")
print(f"Free space on Drive: {stat.free / (1024**3):.2f} GB")


Clone StarGAN repo to access AFHQ images.

In [ ]:
# --- Fresh clone of repo ---
%cd /content
!rm -rf stargan-v2
!git clone https://github.com/clovaai/stargan-v2.git
%cd /content/stargan-v2

# Patch unzip calls inside download.sh to overwrite quietly
!sed -i 's/unzip -q/unzip -oq/g' download.sh

# Make download.sh safe to execute (handle CRLF + perms)
!sed -i -e 's/\r$//' download.sh
!chmod +x download.sh

# Runtime-friendly deps (headless OpenCV avoids GUI conflicts)
!pip -q install munch pyyaml ffmpeg-python scikit-image opencv-python-headless pillow tqdm scipy matplotlib

# --- Download AFHQ ONLY ---
! rm -rf data/afhq data/afhq-v2 # explicitly ensure no mixing
!bash download.sh afhq-dataset
!bash download.sh pretrained-network-afhq


Set-up directories in both Google drive and Colab.

In [ ]:
# Create MyDrive and Colab Directories
from pathlib import Path
import os, shutil, tarfile, zipfile, random, hashlib, io, math, sys
from collections import defaultdict, Counter
import numpy as np, pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 6)
random.seed(42); np.random.seed(42)

# Central dirs
DRIVE_ROOT   = Path("/content/drive/MyDrive/deepfake_project")
PROGRAMS_DIR = DRIVE_ROOT / "programs"
IMAGES_DIR   = DRIVE_ROOT / "images"        # real + fake live here
OUTPUT_DIR   = DRIVE_ROOT / "output"        # csv reports, logs, plots
ZIPS_DIR     = DRIVE_ROOT / "zips"          # zipped exports

WORK_DIR     = Path("/content/stargan_v2_work")   # local scratch in Colab
STARGAN_DIR  = Path("/content/stargan-v2")        # repo checkout

# Techniques (if/when you balance by family later)
TECHNIQUES = ["stargan","fake_copy_move","fake_inpaint","fake_postproc","fake_splicing","fake_swap_like"]

# Ensure structure on Drive
for d in [PROGRAMS_DIR, IMAGES_DIR, OUTPUT_DIR, ZIPS_DIR, WORK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for sub in [
    "real_all_raw", "real_all_dedup", "fake_all_raw",
    "dataset/train/real", "dataset/train/fake",
    "dataset/test/real",  "dataset/test/fake",
    "samples"
]:
    (IMAGES_DIR / sub).mkdir(parents=True, exist_ok=True)

(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
print("Folders ready under:", DRIVE_ROOT)


Download AFHQ images and StarGAN model weights into Colab.

In [ ]:
# Download AFHQ Images and Model Weights

!rm -rf data/afhq
!rm -f data/afhq.zip

!bash download.sh afhq-dataset

!bash download.sh pretrained-network-afhq

# Download Check
!ls -lah data | sed -n '1,120p' || true
!ls -lah expr/checkpoints/afhq | sed -n '1,120p' || true


Create a zip file of all of the images and copy into Google Drive (My Drive).

Display 3 random images each of cat, dog and wild classes.

In [ ]:
# === Show a random sample: 3 images each for cat, dog, wild ===
from pathlib import Path
import random, os
import matplotlib.pyplot as plt
import cv2

DATA_ROOT = Path("/content/stargan-v2/data")
CLASSES   = ["cat", "dog", "wild"]
EXTS      = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# Set a seed if you want reproducible samples; comment this out for fresh randomness each run
# random.seed(42)

def find_images_for(species: str):
    # Search any folder named exactly the species anywhere under DATA_ROOT (e.g., train/cat, val/cat, test/cat)
    candidates = []
    for d in DATA_ROOT.rglob(species):
        if d.is_dir():
            for p in d.iterdir():
                if p.is_file() and p.suffix.lower() in EXTS:
                    candidates.append(p)
    return candidates

# Collect 3 random images per class (or fewer if not enough exist)
samples_by_class = {}
for cls in CLASSES:
    imgs = find_images_for(cls)
    if not imgs:
        print(f"[WARN] No images found for class '{cls}' under {DATA_ROOT}")
        samples_by_class[cls] = []
        continue
    k = min(3, len(imgs))
    samples_by_class[cls] = random.sample(imgs, k=k)

# Plot 3 x 3 grid
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(14, 10))
for r, cls in enumerate(CLASSES):
    row_paths = samples_by_class.get(cls, [])
    for c in range(3):
        ax = axes[r, c]
        ax.axis("off")
        if c < len(row_paths):
            p = row_paths[c]
            # Read with OpenCV and convert BGR->RGB
            img = cv2.imread(str(p))
            if img is None:
                ax.set_title(f"{cls}: [unreadable]\n{p.name}", fontsize=9)
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(f"{cls} • {p.name}", fontsize=10)
        else:
            ax.set_title(f"{cls}: [no image]", fontsize=9)

plt.tight_layout()
plt.show()


Dedupe AFHQ images based on Name and SHA1. Show duplicate images.

In [ ]:
# ===========================
# Dedupe AFHQ — Remove duplicates by Name, SHA1, and pHash (integrated)
# ===========================
# What it does (per species: cat/dog/wild):
#  1) Collect AFHQ images (train/val only, from data/afhq)
#  2) Dedup A: same filename (keeps first seen)
#  3) Dedup B: exact dup by SHA-1 (keeps first hash seen)
#  4) Dedup C: near-dup by perceptual hash (pHash) clustering (keeps 1 per cluster)
#  5) Split 70/30 train/test and copy into data/afhq_all/{train,test}/{species}
#  6) Write manifests + reports + sanity checks


import os, hashlib, random, shutil, json
from pathlib import Path
from datetime import datetime
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# ---- install once in your notebook ----
# !pip -q install ImageHash
import imagehash

# ---------------- Config ----------------
random.seed(42)
np.random.seed(42)

MAX_SHOW  = 30      # max duplicate pairs to visualize
MAX_PRINT = 50      # max lines to print in console reports
PREFER_SYMLINKS = False
DELETE_DUP_FILES = False   # set True to physically delete duplicate source files

# pHash near-dup config
PHASH_DEDUP      = True
PHASH_SIZE       = 16      # 16 => 256-bit pHash (more stable than 8)
HAMMING_T        = 8       # <=6 strict; 8 more sensitivity
PREFIX_HEX       = 6       # bucket by first N hex chars to avoid O(N^2)
PHASH_KEEP_POLICY = "prefer_larger"  # "prefer_larger" or "prefer_first"

root = Path.cwd()
src_roots = [root / "data" / "afhq"]
src_roots = [p for p in src_roots if p.exists()]
if not src_roots:
    raise FileNotFoundError("AFHQ dataset not found at data/afhq. Did download.sh afhq-dataset complete?")
print("[DISCOVERY] Using roots:", [str(p) for p in src_roots])

# Clean outputs so nothing mixes across runs
!rm -rf data/afhq_all tmp_refs

dst_root    = root / "data" / "afhq_all"
species     = ["cat","dog","wild"]
splits      = ["train","test"]
train_ratio = 0.70

# Where to place the zipped copies
ZIPS_DIR  = Path("/content/drive/MyDrive/deepfake_project/zips")
ZIPS_DIR.mkdir(parents=True, exist_ok=True)

EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

run_cfg = {
    "source_dataset": "afhq_only",
    "dedup_name": True,
    "dedup_sha1": True,
    "dedup_phash": PHASH_DEDUP,
    "phash_size": PHASH_SIZE,
    "phash_hamming_threshold": HAMMING_T,
    "phash_prefix_hex_bucket": PREFIX_HEX,
    "phash_keep_policy": PHASH_KEEP_POLICY,
    "delete_dup_files": DELETE_DUP_FILES,
    "prefer_symlinks": PREFER_SYMLINKS,
    "train_ratio": train_ratio,
    "seed": 42
}
(root/"program1_run_config.json").write_text(json.dumps(run_cfg, indent=2))
print("[WRITE] program1_run_config.json")

# ---------------- Helpers ----------------
def sha1(fp: Path, chunk=1<<20):
    h = hashlib.sha1()
    with open(fp, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    if PREFER_SYMLINKS:
        try:
            os.symlink(src, dst)
            return
        except OSError:
            pass
    shutil.copy2(src, dst)

def show_pairs(pairs, title="Duplicates", max_show=MAX_SHOW):
    if not pairs:
        print(f"[DISPLAY] No {title.lower()} to display.")
        return
    k = min(len(pairs), max_show)
    rows, cols = k, 2
    fig, axes = plt.subplots(rows, cols, figsize=(8, 3*rows))
    if rows == 1:
        kept, dup, note = pairs[0]
        for j, pth in enumerate([kept, dup]):
            ax = axes[j]
            try:
                im = Image.open(pth).convert("RGB")
                ax.imshow(im)
                ax.set_title(("KEPT","DUP")[j] + f": {Path(pth).name}\n{note}", fontsize=9)
            except Exception as e:
                ax.text(0.5, 0.5, f"Error:\n{pth}\n{e}", ha='center', va='center')
            ax.axis('off')
        plt.suptitle(title, fontsize=12); plt.tight_layout(); plt.show()
        return
    for i in range(k):
        kept, dup, note = pairs[i]
        for j, pth in enumerate([kept, dup]):
            ax = axes[i, j]
            try:
                im = Image.open(pth).convert("RGB")
                ax.imshow(im)
                role = "KEPT" if j == 0 else "DUP"
                ax.set_title(f"{role}: {Path(pth).name}\n{note}", fontsize=9)
            except Exception as e:
                ax.text(0.5, 0.5, f"Error:\n{pth}\n{e}", ha='center', va='center')
            ax.axis('off')
    plt.suptitle(title, fontsize=12); plt.tight_layout(); plt.show()

def zip_and_move(folder: Path, label: str, out_dir: Path) -> Path:
    if not folder.exists() or not any(folder.rglob("*")):
        raise RuntimeError(f"Nothing to zip in {folder}")
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    base_name = Path(f"/content/{label}-{stamp}")   # temp path (no extension)
    archive_path = shutil.make_archive(
        base_name=str(base_name),
        format="zip",
        root_dir=folder.parent,
        base_dir=folder.name
    )
    out_dir.mkdir(parents=True, exist_ok=True)
    final = out_dir / Path(archive_path).name
    shutil.move(archive_path, final)
    print(f"[ZIP] Wrote: {final} ({final.stat().st_size / (1024**3):.2f} GB)")
    return final

# ----- pHash helpers -----
def safe_phash_hex(p: Path, phash_size=PHASH_SIZE):
    try:
        img = Image.open(p).convert("RGB")
        return str(imagehash.phash(img, hash_size=phash_size))
    except Exception:
        return None

def hamming_hex(a_hex: str, b_hex: str) -> int:
    return int((int(a_hex, 16) ^ int(b_hex, 16)).bit_count())

def choose_keeper(p1: Path, p2: Path) -> Path:
    if PHASH_KEEP_POLICY == "prefer_larger":
        # keep larger file (often better quality)
        return p1 if p1.stat().st_size >= p2.stat().st_size else p2
    # stable fallback
    return p1

def phash_dedupe_within_species(paths, species_name: str):
    """
    Input: list[Path] already name+sha1 deduped
    Output: kept_paths (list[Path]) + report DFs
    """
    rows = []
    for p in paths:
        rows.append({
            "species": species_name,
            "path": str(p),
            "filename": p.name,
            "size_bytes": p.stat().st_size if p.exists() else None,
            "phash_hex": safe_phash_hex(p)
        })
    mf = pd.DataFrame(rows)
    mf_ok = mf.dropna(subset=["phash_hex"]).reset_index(drop=True)

    # bucket by prefix for speed
    mf_ok["bucket"] = mf_ok["phash_hex"].str[:PREFIX_HEX]
    buckets = defaultdict(list)
    for i, b in enumerate(mf_ok["bucket"].tolist()):
        buckets[b].append(i)

    edges = []
    adj = defaultdict(list)

    checked = 0
    for b, idxs in buckets.items():
        if len(idxs) < 2:
            continue
        for ii in range(len(idxs)):
            i = idxs[ii]
            for jj in range(ii + 1, len(idxs)):
                j = idxs[jj]
                checked += 1
                d = hamming_hex(mf_ok.at[i, "phash_hex"], mf_ok.at[j, "phash_hex"])
                if d <= HAMMING_T:
                    edges.append((i, j, d))
                    adj[i].append(j)
                    adj[j].append(i)

    # connected components
    visited = set()
    clusters = []
    for node in range(len(mf_ok)):
        if node in visited or node not in adj:
            continue
        q = deque([node]); visited.add(node)
        comp = []
        while q:
            u = q.popleft()
            comp.append(u)
            for v in adj[u]:
                if v not in visited:
                    visited.add(v)
                    q.append(v)
        clusters.append(comp)

    removed_rows = []
    keep_set = set(mf_ok["path"].tolist())

    for cid, comp in enumerate(clusters, start=1):
        comp_paths = [Path(mf_ok.iloc[i]["path"]) for i in comp]
        keeper = comp_paths[0]
        for p in comp_paths[1:]:
            keeper = choose_keeper(keeper, p)
        for p in comp_paths:
            if p != keeper and str(p) in keep_set:
                keep_set.remove(str(p))
                removed_rows.append({
                    "species": species_name,
                    "cluster_id": cid,
                    "kept": str(keeper),
                    "removed": str(p),
                    "kept_file": keeper.name,
                    "removed_file": p.name
                })

    edges_df = pd.DataFrame(edges, columns=["i","j","hamming"])
    removed_df = pd.DataFrame(removed_rows)

    print(f"[pHash:{species_name}] hashed={len(mf_ok):,}  checked_pairs={checked:,}  clusters={len(clusters):,}  removed={len(removed_df):,}  kept={len(keep_set):,}")
    return [Path(x) for x in keep_set], mf_ok, edges_df, removed_df

# ===========================
# 1) Collect AFHQ
# ===========================
all_paths = []  # list of tuples (species, Path)
pre_counts = {s: 0 for s in species}

for s in species:
    for src in src_roots:
        for split_guess in ["train","val",""]:
            d = src/split_guess/s if split_guess else src/s
            if d.exists():
                found = [(s, p) for p in d.rglob("*") if p.is_file() and p.suffix.lower() in EXTS]
                all_paths += found
                pre_counts[s] += len(found)

print(f"[DISCOVERY] Collected {len(all_paths):,} files across roots.")
print("[COUNTS: pre-dedupe by species]", pre_counts, "| total:", sum(pre_counts.values()))

# ===========================
# 2) Dedup A: by filename (within species)
# ===========================
seen_names = {s: {} for s in species}
name_dups, kept_after_name = [], {s: [] for s in species}

for s, p in all_paths:
    nm = p.name
    if nm in seen_names[s]:
        name_dups.append({"species": s, "dup": str(p), "kept": str(seen_names[s][nm])})
    else:
        seen_names[s][nm] = p
        kept_after_name[s].append(p)

pd.DataFrame(name_dups).to_csv(root/"name_dups.csv", index=False)
print(f"[NAME] Duplicates removed: {len(name_dups):,} (wrote name_dups.csv)")

# ===========================
# 3) Dedup B: by SHA-1 (within species)
# ===========================
hash_maps = {s: {} for s in species}
sha_dups, sha_errors = [], []

for s in species:
    for p in kept_after_name[s]:
        try:
            h = sha1(p)
        except Exception as e:
            sha_errors.append({"species": s, "path": str(p), "error": repr(e)})
            continue
        if h in hash_maps[s]:
            sha_dups.append({"species": s, "dup": str(p), "kept": str(hash_maps[s][h]), "sha1": h})
        else:
            hash_maps[s][h] = p

pd.DataFrame(sha_dups).to_csv(root/"sha_dups.csv", index=False)
pd.DataFrame(sha_errors).to_csv(root/"sha1_errors.csv", index=False)
print(f"[SHA1] Duplicates removed: {len(sha_dups):,} (wrote sha_dups.csv)")
print(f"[SHA1] Errors computing hash: {len(sha_errors):,} (wrote sha1_errors.csv)")

# Optionally remove duplicate files at the source
if DELETE_DUP_FILES:
    to_delete = [Path(rec["dup"]) for rec in name_dups] + [Path(rec["dup"]) for rec in sha_dups]
    deleted = 0
    for fp in to_delete:
        try:
            fp.unlink()
            deleted += 1
        except Exception:
            pass
    print(f"[DELETE] Removed {deleted} duplicate files from source locations.")

# Final kept set after SHA-1 (unique by SHA-1 within species)
final_kept = {s: sorted(set(hash_maps[s].values())) for s in species}

post_sha_counts = {s: len(final_kept[s]) for s in species}
print("\n[COUNTS: post-SHA1 by species]", post_sha_counts, "| total:", sum(post_sha_counts.values()))

# ===========================
# 4) Dedup C: pHash near-duplicates (within species) — REMOVE
# ===========================
phash_manifests = []
phash_edges_all = []
phash_removed_all = []

if PHASH_DEDUP:
    for s in species:
        kept_paths, mf_ok, edges_df, removed_df = phash_dedupe_within_species(list(final_kept[s]), s)
        final_kept[s] = sorted(kept_paths)

        mf_ok.to_csv(root/f"phash_manifest_{s}.csv", index=False)
        edges_df.to_csv(root/f"phash_edges_{s}.csv", index=False)
        removed_df.to_csv(root/f"phash_removed_{s}.csv", index=False)

        phash_manifests.append(mf_ok)
        phash_edges_all.append(edges_df.assign(species=s))
        phash_removed_all.append(removed_df)

    # combined reports
    pd.concat(phash_manifests, ignore_index=True).to_csv(root/"phash_manifest_all_species.csv", index=False)
    pd.concat(phash_edges_all, ignore_index=True).to_csv(root/"phash_edges_all_species.csv", index=False)
    pd.concat(phash_removed_all, ignore_index=True).to_csv(root/"phash_removed_all_species.csv", index=False)

    post_phash_counts = {s: len(final_kept[s]) for s in species}
    print("\n[COUNTS: post-pHash by species]", post_phash_counts, "| total:", sum(post_phash_counts.values()))
else:
    print("[pHash] PHASH_DEDUP=False (skipping near-duplicate removal)")

# ===========================
# 5) Split (70/30) + write afhq_all dataset
# ===========================
for split in splits:
    for s in species:
        (dst_root/split/s).mkdir(parents=True, exist_ok=True)

assignments = []
for s in species:
    items = list(final_kept[s])
    random.shuffle(items)
    k_train = int(round(train_ratio * len(items)))
    train_set = items[:k_train]
    test_set  = items[k_train:]

    for split, group in [("train", train_set), ("test", test_set)]:
        for p in group:
            try:
                h = sha1(p)
            except Exception:
                h = None
            out_name = (f"{h[:8]}__{p.name}") if h else p.name
            dst = dst_root/split/s/out_name
            link_or_copy(p, dst)
            assignments.append({
                "source_dataset": "afhq",
                "split": split,
                "species": s,
                "src": str(p),
                "dst": str(dst),
                "filename": dst.name,
                "sha1": h
            })

manifest_df = pd.DataFrame(assignments)
manifest_path = root/"afhq_all_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
print("[WRITE] Manifest:", manifest_path)

# ===========================
# 6) Sanity checks
# ===========================
if "filename" not in manifest_df.columns:
    raise KeyError("manifest_df missing 'filename' column (expected filename=dst.name).")

dup_mask = manifest_df["filename"].duplicated(keep=False)
if dup_mask.any():
    dups = manifest_df.loc[dup_mask].sort_values(["filename","split","species"])
    dups_path = root/"manifest_filename_dups.csv"
    dups.to_csv(dups_path, index=False)
    raise ValueError(f"Duplicate filenames detected in manifest. See: {dups_path}")

print("[MANIFEST] rows:", len(manifest_df))
print("[MANIFEST] unique filenames:", manifest_df["filename"].nunique())

# counts on disk
result_counts = {split: {s: 0 for s in species} for split in splits}
for split in splits:
    for s in species:
        files = [x for x in (dst_root/split/s).glob("*") if x.is_file()]
        result_counts[split][s] = len(files)

print("\n[DATASET COUNTS]")
for split in splits:
    print(split, result_counts[split], "| total:", sum(result_counts[split].values()))

# ===========================
# 7) Zip train/ and test/ and move to Drive
# ===========================
train_zip = zip_and_move(dst_root/"train", label="afhq_all-train", out_dir=ZIPS_DIR)
test_zip  = zip_and_move(dst_root/"test",  label="afhq_all-test",  out_dir=ZIPS_DIR)
print("[DONE] Zipped archives moved to:", ZIPS_DIR)

Create two random train sets and 1 random test set of images to apply StarGAN for GAN deepfakes. This process is time-consuming and requires heavy processing.

The output of the StarGAN is a tile image which has to be cut or parsed to get the individual deepfake images. There are two tile images for train and 1 tile image for test.

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
# Make 5 separate StarGAN v2 runs for TRAIN (distinct refs/output), plus 3 for TEST
from pathlib import Path
import os, random, shutil

repo = Path("/content/stargan-v2")
base = repo
real_root = repo/"data/afhq_all"

def make_ref_tree(split, tag, n_per_domain=5, seed=None):
    if seed is not None:
        random.seed(seed)

    ref_root = repo/f"tmp_refs/{split}_{tag}"
    for dom in ["cat","dog","wild"]:
        d = ref_root/dom
        d.mkdir(parents=True, exist_ok=True)
        for f in d.glob("*"): f.unlink()

        imgs = [p for p in (real_root/split/dom).glob("*")
                if p.suffix.lower() in [".jpg",".jpeg",".png"]]
        random.shuffle(imgs)
        for p in imgs[:min(n_per_domain, len(imgs))]:
            try:
                os.symlink(p, d/p.name)
            except OSError:
                shutil.copy2(p, d/p.name)
    return ref_root

# ===== Build separate reference sets =====
# different seeds to ensure different grids
print("Created refs at:", make_ref_tree("train", "run1", n_per_domain=5, seed=123))
print("Created refs at:", make_ref_tree("train", "run2", n_per_domain=5, seed=456))
print("Created refs at:", make_ref_tree("train", "run3", n_per_domain=5, seed=789))
print("Created refs at:", make_ref_tree("train", "run4", n_per_domain=5, seed=234))
print("Created refs at:", make_ref_tree("train", "run5", n_per_domain=5, seed=567))


print("Created refs at:", make_ref_tree("test",  "run1", n_per_domain=5, seed=123))
print("Created refs at:", make_ref_tree("test",  "run2", n_per_domain=5, seed=456))
print("Created refs at:", make_ref_tree("test",  "run3", n_per_domain=5, seed=789))



from pathlib import Path
import subprocess

repo = Path("/content/stargan-v2")

runs = [
    # TRAIN (5)
    ("train_run1", "data/afhq_all/train", "tmp_refs/train_run1", 987),
    ("train_run2", "data/afhq_all/train", "tmp_refs/train_run2", 654),
    ("train_run3", "data/afhq_all/train", "tmp_refs/train_run3", 321),
    ("train_run4", "data/afhq_all/train", "tmp_refs/train_run4", 1321),
    ("train_run5", "data/afhq_all/train", "tmp_refs/train_run5", 5465),

    # TEST (3)
    ("test_run1",  "data/afhq_all/test",  "tmp_refs/test_run1",  6789),
    ("test_run2",  "data/afhq_all/test",  "tmp_refs/test_run2",  9132),
    ("test_run3",  "data/afhq_all/test",  "tmp_refs/test_run3",  7543),
]

for run_name, src_dir, ref_dir, seed in runs:
    cmd = [
        "python", "main.py",
        "--mode", "sample",
        "--num_domains", "3",
        "--num_workers", "0",
        "--w_hpf", "0",
        "--resume_iter", "100000",
        "--seed", str(seed),
        "--checkpoint_dir", "expr/checkpoints/afhq",
        "--result_dir", f"expr/results/afhq_all_fake_stargan/{run_name}",
        "--src_dir", str(repo/src_dir),
        "--ref_dir", str(repo/ref_dir),
        "--result_dir", str(repo/f"expr/results/afhq_all_fake_stargan/{run_name}"),
    ]
    print(f"\n=== Running {run_name} (seed={seed}) ===")
    subprocess.run(cmd, cwd=str(repo), check=True)

In [ ]:
# Make sure refs are actually different for each run
from pathlib import Path

base = Path("/content/stargan-v2/tmp_refs")

for tag in ["train_run1","train_run2","train_run3","train_run4","train_run5","test_run1","test_run2","test_run3"]:
    print(f"\n{tag}:")
    for dom in ["cat","dog","wild"]:
        files = sorted((base/tag/dom).glob("*"))
        print(f"  {dom}: {len(files)}  sample={[f.name for f in files[:3]]}")

In [ ]:
# === Zip all StarGAN tile images and move to Drive ===
from pathlib import Path
from datetime import datetime
import shutil, os

ZIPS_DIR = Path("/content/drive/MyDrive/deepfake_project/zips")
ZIPS_DIR.mkdir(parents=True, exist_ok=True)

RESULT_ROOT = Path("/content/stargan-v2/expr/results/afhq_all_fake_stargan")
if not RESULT_ROOT.exists():
  raise FileNotFoundError(f"StarGAN results not found at: {RESULT_ROOT}")

# Stage only images to keep archive small
STAGE_DIR = Path("/content/_tiles_stage")
if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)
STAGE_DIR.mkdir(parents=True, exist_ok=True)

def is_img(p: Path) -> bool:
    return p.is_file() and p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

copied = 0
for p in RESULT_ROOT.rglob("*"):
    if is_img(p):
        rel = p.relative_to(RESULT_ROOT)     # keep subfolder structure
        dst = STAGE_DIR / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dst)
        copied += 1

if copied == 0:
    raise RuntimeError(f"No image tiles found under {RESULT_ROOT}")

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
archive_base = Path(f"/content/afhq_all_fake_stargan_tiles-{stamp}")  # no extension here
zip_path = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=str(STAGE_DIR.parent),
    base_dir=STAGE_DIR.name
)

final_path = ZIPS_DIR / Path(zip_path).name
shutil.move(zip_path, final_path)

# Optional: clean up staging
shutil.rmtree(STAGE_DIR, ignore_errors=True)

size_gb = final_path.stat().st_size / (1024**3)
print(f"✅ Zipped {copied:,} tile images to: {final_path}")
print(f"   Size: {size_gb:.2f} GB")


This is to show an example of what tile looks like. The first row and column are real images with blended image occurring in the middle.

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

RESULT_ROOT = Path("/content/stargan-v2/expr/results/afhq_all_fake_stargan")
exts = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

def find_first_image_dir(root: Path):
    # find first directory that contains at least one image
    for d in sorted([p for p in root.iterdir() if p.is_dir()]):
        for p in d.rglob("*"):
            if p.is_file() and p.suffix.lower() in exts:
                return d, p
    return None, None

run_dir, img_path = find_first_image_dir(RESULT_ROOT)

if not img_path:
    raise FileNotFoundError(f"No images found under {RESULT_ROOT}. Did StarGAN complete any run?")

print("Run folder:", run_dir.name)
print("Showing:", img_path)

img = Image.open(img_path).convert("RGB")
plt.figure(figsize=(8,8))
plt.imshow(img)
plt.axis("off")
plt.show()
